In [1]:
import seaborn as sns
import requests
import pandas as pd
import numpy as np
from datetime import datetime
from astropy.time import Time
from LINZ.CORS_Timeseries import TimeseriesList, CoordApiTimeseries
import os 
from statsmodels.tsa.stattools import acf
import matplotlib.pyplot as plt
import re
from typing import Dict

In [2]:
# Define time range
start = datetime(2023, 1, 14)
end = datetime(2023, 2, 4)

In [3]:
# These are all of the current Syngery sites, copy in the ones you want.
#SYcodes=['SYAS','SYCC','SYCM','SYDF','SYHA','SYHW','SYM2','SYMV','SYNL','SYQN','SYT2','SYTA','SYTS','SYWM','SYET','SYBM','SYRD','SYLN','SYWK','SYPP','SYP2', 'SYOI', 'SYP3', 'SYPO', 'SYPR', 'SYMM', 'SYGR']
SYcodes=['SYBM']

In [4]:
# Calculate crd_epoch
crd_epoch = end + (start - end) / 2
astropy_time_object = Time(crd_epoch, format='datetime')
crd_epoch_decimal_year = astropy_time_object.decimalyear

In [14]:
# --- Clear, human-readable labels for solution types ---
SOLUTION_LABEL_MAP = {
    'd20f_52_code_B' :   'GPS'
    #'p20r_54_code' :    'GPS+Galileo+GLONASS'
    }

# Initialise a dictionary to store DataFrames for each solution type
daily_avg_dfs = {
    # Keep keys as your original solution types for API calls and grouping
    'd20f_52_code_B': {}
    #'p20r_54_code': {}
}

# Loop through SYcodes and solution types
for code in SYcodes:
    for sol_type in daily_avg_dfs.keys():
        try:
            print(f"Processing station: {code} with solution type: {sol_type}")
            ts = CoordApiTimeseries(code, solutiontype=sol_type, after=start, before=end)
            dates, xyz = ts.withoutOutliers().getObs(enu=False)

            df_obs = pd.DataFrame(xyz, columns=['x', 'y', 'z'], index=dates)
            df_obs['date'] = df_obs.index.strftime('%Y-%m-%d')
            df_obs['station'] = code

            # Apply the human-readable label
            df_obs['solutiontype'] = SOLUTION_LABEL_MAP.get(sol_type, sol_type)

            daily_avg_dfs[sol_type][code] = df_obs

        except Exception as e:
            print(f"Failed to process station {code} with solution type {sol_type}. Error: {e}")

# Combine all DataFrames into one
SY_data = pd.concat(
    [pd.concat(dfs.values(), axis=0) for dfs in daily_avg_dfs.values()],
    axis=0
)

# Sort alphabetically by station, date, and solution type
SY_data.sort_values(by=["station", "date", "solutiontype"], inplace=True)

# Print confirmation
print("Download complete ✅")
print("SY_data DataFrame created with shape:", SY_data.shape)
print("Columns:", SY_data.columns.tolist())
print("Solution types present:", sorted(SY_data['solutiontype'].unique()))

Processing station: SYBM with solution type: d20f_52_code_B
Download complete ✅
SY_data DataFrame created with shape: (20, 6)
Columns: ['x', 'y', 'z', 'date', 'station', 'solutiontype']
Solution types present: ['GPS']


In [15]:
# Group SY_data by station and solutiontype
group_dataframes = {
    f"{station}_{sol_type}": group
    for (station, sol_type), group in SY_data.groupby(["station", "solutiontype"])
}

In [16]:
# Define expected converted DataFrame names
converted_columns = [
    'x', 'y', 'z', 'date', 'station',
    'nztm_easting', 'nztm_northing', 'nzvd2016_elev'
]

# Initialize empty dictionary with expected keys
empty_converted_dfs = {
    f"{name}_converted": pd.DataFrame(columns=converted_columns)
    for name in group_dataframes.keys()
}

In [17]:
converted_group_dataframes = {}

def convert_coordinates(input_crds, crd_epoch_decimal_year):
    occ_api = "https://www.geodesy.linz.govt.nz/api/conversions/v1/convert-to"
    coordlist = {
        "crs": "LINZ:ITRF2020_XYZ",
        "coordinateOrder": ["geocentricX", "geocentricY", "geocentricZ"],
        "coordinateEpoch": crd_epoch_decimal_year,
        "coordinates": input_crds
    }
    params = {"crs": "LINZ:NZTM/NZVD2016"}

    response = requests.post(occ_api, params=params, json=coordlist)
    if response.status_code == 200:
        converted = response.json()
        return converted['coordinateList']['coordinates']
    else:
        print(f"❌ API conversion failed with status {response.status_code}")
        print(f"🔍 Error details: {response.text}")
        return []

In [18]:
for name, original_df in group_dataframes.items():
    print(f"\n🔄 Processing DataFrame: {name}")

    # Always start with a fresh copy
    df = original_df.copy(deep=True)

    # Identify and print NaN values
    nan_values = df[df.isna().any(axis=1)]
    if not nan_values.empty:
        print(f" - Found {len(nan_values)} NaN rows")

    # Identify and print Infinity values
    inf_values = df[(df == np.inf) | (df == -np.inf)].dropna(how='all')
    if not inf_values.empty:
        print(f" - Found {len(inf_values)} Inf rows")

    # Filter out invalid rows
    df_filtered = df.dropna(subset=['x', 'y', 'z']).copy()
    df_filtered = df_filtered[
        (df_filtered['x'] != np.inf) & (df_filtered['x'] != -np.inf) &
        (df_filtered['y'] != np.inf) & (df_filtered['y'] != -np.inf) &
        (df_filtered['z'] != np.inf) & (df_filtered['z'] != -np.inf)
    ]

    if df_filtered.empty:
        print(f"⚠️ Skipping {name} — no valid coordinates after filtering.")
        continue

    # Prepare coordinates
    input_crds = df_filtered[['x', 'y', 'z']].values.tolist()

    # Convert coordinates
    converted_coords = convert_coordinates(input_crds, crd_epoch_decimal_year)

    # Add converted coordinates to DataFrame
    if converted_coords:
        df_filtered[['nztm_easting', 'nztm_northing', 'nzvd2016_elev']] = pd.DataFrame(
            converted_coords, index=df_filtered.index
        )

        # Store in the pre-initialised DataFrame
        converted_name = f"{name}_converted"
        if converted_name in empty_converted_dfs:
            empty_converted_dfs[converted_name] = df_filtered
            print(f"✅ {converted_name} filled and updated.")
        else:
            print(f"⚠️ {converted_name} not found in empty_converted_dfs. Skipping storage.")
    else:
        print(f"❌ Conversion failed for {name}. No coordinates added.")


🔄 Processing DataFrame: SYBM_GPS
✅ SYBM_GPS_converted filled and updated.


In [19]:
# Rename for clarity — this is now the final dictionary of converted DataFrames
converted_group_dataframes = empty_converted_dfs
del empty_converted_dfs  # Optional: clean up the old name if no longer needed

In [20]:
def plot_available_solutions(
    converted_group_dataframes: Dict[str, pd.DataFrame],
    window_days: int = 30,
    min_solutions_per_station: int = 1,
    verbose: bool = True,
    save_folder: str = "Synergy_timeseries",
    save_pdf: bool = False
):
    """
    Plot GNSS time series for each station and save figures to a folder.
    Fixes scientific notation and ensures clean filenames.
    Breaks the time series whenever a calendar day is missing (gap > 1 day).
    """

    # Create folder if it doesn't exist
    os.makedirs(save_folder, exist_ok=True)

    # Regex to extract network and solution
    key_re = re.compile(r"^(?P<network>[^_]+)_(?P<solution>.+?)_converted$")
    ns_map: Dict[str, Dict[str, pd.DataFrame]] = {}
    unmatched = []

    # Discover networks and solutions
    for key, df in converted_group_dataframes.items():
        m = key_re.match(key)
        if not m:
            unmatched.append(key)
            continue
        net = m.group("network")
        sol = m.group("solution")
        ns_map.setdefault(net, {})[sol] = df.copy()

    if verbose:
        print("=== DISCOVERY ===")
        print(f"Total dict keys: {len(converted_group_dataframes)}")
        print("Networks found:", sorted(ns_map.keys()))

    # Helper: plot a line that breaks at gaps > 1 day
    def _plot_broken(ax, data, x_col, y_col, colour, linestyle='-', linewidth=1, alpha=1.0, label=None):
        """
        Plot line segments, splitting wherever date gaps > 1 day occur.
        Legend label is applied only to the first segment if provided.
        """
        g = data[[x_col, y_col]].dropna().sort_values(x_col)
        if g.empty:
            return
        # Identify where a gap occurs between consecutive samples
        gap_mask = g[x_col].diff().gt(pd.Timedelta(days=1)).fillna(False)
        seg_id = gap_mask.cumsum()
        first = True
        for _, seg in g.groupby(seg_id):
            ax.plot(
                seg[x_col], seg[y_col],
                linestyle=linestyle, linewidth=linewidth, color=colour, alpha=alpha,
                label=(label if first else None)
            )
            first = False

    # Process each network
    for network, sols_dict in sorted(ns_map.items()):
        if verbose:
            print(f"\n=== NETWORK: {network} ===")

        # Clean frames
        for sol, df in list(sols_dict.items()):
            required_cols = ("date", "station", "nzvd2016_elev")
            missing = [c for c in required_cols if c not in df.columns]
            if missing:
                sols_dict.pop(sol)
                continue

            df['date'] = pd.to_datetime(df['date'], errors='coerce')
            df.dropna(subset=['date'], inplace=True)
            df['station'] = df['station'].astype(str).str.strip()  # FIXED
            sols_dict[sol] = df

        if not sols_dict:
            continue

        stations_union = sorted(set().union(*[set(df['station']) for df in sols_dict.values()]))
        palette = dict(zip(sorted(sols_dict.keys()), sns.color_palette("colorblind", n_colors=len(sols_dict))))

        # Per-station plotting
        for station_name in stations_union:
            station_frames = {}
            for sol, df in sols_dict.items():
                d = (
                    df.loc[df['station'] == station_name]
                      .sort_values('date')
                      .set_index('date')
                      .copy()
                )
                if d.empty:
                    continue

                d['running_mean']  = d['nzvd2016_elev'].rolling(f'{window_days}D', min_periods=1).mean()
                d['detrended_mm']  = (d['nzvd2016_elev'] - d['running_mean']) * 1000
                d['elevation']     = d['nzvd2016_elev']
                d['solutiontype']  = sol
                station_frames[sol] = d

            if len(station_frames) < min_solutions_per_station:
                continue

            combined_df = pd.concat(station_frames.values()).reset_index()

            # Plot
            sns.set(style="whitegrid")
            fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

            # Elevation: raw (dashed) and running mean (solid)
            for sol_key, colour in palette.items():
                g = combined_df[combined_df['solutiontype'] == sol_key].sort_values('date')
                if g.empty:
                    continue
                _plot_broken(ax[0], g, 'date', 'elevation',     colour, linestyle='--', linewidth=1, alpha=0.6, label=None)
                _plot_broken(ax[0], g, 'date', 'running_mean',  colour, linestyle='-',  linewidth=2, alpha=1.0,  label=sol_key)

            ax[0].set_title(f'Elevation and Running Mean — {network} / {station_name}')
            ax[0].set_ylabel('Elevation (m)')
            ax[0].legend(title='Solution Type')
            ax[0].grid(True)
            # Fix scientific notation
            from matplotlib.ticker import FormatStrFormatter
            ax[0].ticklabel_format(useOffset=False, axis='y')
            ax[0].yaxis.set_major_formatter(FormatStrFormatter('%.3f'))

            # Residuals
            for sol_key, colour in palette.items():
                g = combined_df[combined_df['solutiontype'] == sol_key].sort_values('date')
                if g.empty:
                    continue
                _plot_broken(ax[1], g, 'date', 'detrended_mm', colour, linestyle='-', linewidth=1, alpha=1.0, label=sol_key)

            ax[1].axhline(0, color='grey', linestyle='--', linewidth=1)
            ax[1].set_title(f'Detrended Residuals — {network} / {station_name}')
            ax[1].set_xlabel('Date')
            ax[1].set_ylabel('Residuals (mm)')
            ax[1].legend(title='Solution Type')
            ax[1].grid(True)

            plt.xticks(rotation=45)
            plt.tight_layout()

            # Safe filename
            safe_station = re.sub(r'[^A-Za-z0-9_-]', '_', station_name)
            filename_png = f"{network}_{safe_station}.png"
            filepath_png = os.path.join(save_folder, filename_png)

            # Save PNG
            plt.savefig(filepath_png, dpi=300)
            if save_pdf:
                filename_pdf = f"{network}_{safe_station}.pdf"
                filepath_pdf = os.path.join(save_folder, filename_pdf)
                plt.savefig(filepath_pdf)

            plt.close(fig)

            if verbose:
                print(f"Saved plot: {filepath_png}")
                if save_pdf:
                    print(f"Saved plot: {filepath_pdf}")

plot_available_solutions(converted_group_dataframes, window_days=30, min_solutions_per_station=1, verbose=True)


=== DISCOVERY ===
Total dict keys: 1
Networks found: ['SYBM']

=== NETWORK: SYBM ===
Saved plot: Synergy_timeseries\SYBM_SYBM.png


In [21]:
import pandas as pd

def detect_misbehaviour_gap_aware(
    converted_group_dataframes,
    window_days=30,
    threshold_factor=3,
    output_file='misbehaviour_start_dates.csv'
):
    results = []

    for key, df in converted_group_dataframes.items():
        # Extract network and solution type from key for context
        parts = key.split('_')
        network = parts[0]
        solution = '_'.join(parts[1:-1])  # everything except "_converted"

        # Ensure required columns exist
        if not {'date', 'station', 'nzvd2016_elev'}.issubset(df.columns):
            continue

        # Prepare dataframe
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df.dropna(subset=['date'], inplace=True)
        df.sort_values('date', inplace=True)

        for station, s_df in df.groupby('station'):
            s_df = s_df.copy()
            s_df['running_mean'] = s_df['nzvd2016_elev'].rolling(f'{window_days}D', min_periods=1).mean()
            s_df['detrended_mm'] = (s_df['nzvd2016_elev'] - s_df['running_mean']) * 1000

            # Detect gaps: compute day differences
            s_df['day_diff'] = s_df['date'].diff().dt.days
            gap_positions = [i for i, diff in enumerate(s_df['day_diff']) if diff > 7]  # positions as integers

            if gap_positions:
                last_gap_index = gap_positions[-1]
                last_gap_date = s_df.iloc[last_gap_index]['date']
            else:
                last_gap_index = None
                last_gap_date = None

            # Compute rolling std after last gap (or from start if no gap)
            start_idx = last_gap_index + 1 if last_gap_index is not None else 0
            s_df['rolling_std'] = s_df['detrended_mm'].rolling(window=window_days, min_periods=5).std()

            # Baseline: median of first 180 days (before gap)
            baseline_period = s_df[s_df['date'] <= s_df['date'].min() + pd.Timedelta(days=180)]
            baseline_std = baseline_period['rolling_std'].median()

            # Fallback if baseline is NaN or zero
            if pd.isna(baseline_std) or baseline_std == 0:
                baseline_std = 5.0  # assume 5 mm as fallback baseline

            threshold = baseline_std * threshold_factor

            # Search for misbehaviour after last gap
            exceed = s_df.iloc[start_idx:][s_df['rolling_std'] > threshold]

            if not exceed.empty:
                misbehave_date = exceed.iloc[0]['date']
                status = 'detected'
            else:
                misbehave_date = None
                status = 'no_issue'

            results.append({
                'network': network,
                'solution': solution,
                'station': station,
                'start_date': misbehave_date,
                'threshold_mm': round(threshold, 2),
                'last_gap_date': last_gap_date,
                'status': status
            })

    # Convert results to DataFrame and save to CSV
    summary_df = pd.DataFrame(results)
    summary_df.to_csv(output_file, index=False)
    print(f"Saved {len(summary_df)} rows to {output_file}")
    return summary_df

# Example usage:
summary_df = detect_misbehaviour_gap_aware(converted_group_dataframes)
summary_df.head()

Saved 1 rows to misbehaviour_start_dates.csv


,network,solution,station,start_date,threshold_mm,last_gap_date,status
0,SYBM,GPS,SYBM,None,8.83,None,no_issue
